# Xcapit FHE-ML Platform - Insurance: Deteccion de Fraude en Reclamos

## Caso de Uso: Consorcio de Aseguradoras LATAM

Este notebook demuestra el flujo completo de:
1. Registro y configuracion del consorcio de aseguradoras
2. Generacion de datos sinteticos de reclamos
3. Encriptacion con FHE (CKKS)
4. Votacion commit-reveal para gobernanza
5. Entrenamiento de modelo de deteccion de fraude
6. Predicciones sobre reclamos encriptados

### Escenario
Tres aseguradoras de LATAM (Argentina, Chile, Mexico) quieren colaborar para detectar fraude en reclamos sin compartir datos sensibles de clientes.

## 1. Setup e Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.getcwd())))

import numpy as np
import pandas as pd
import hashlib
import secrets
from datetime import datetime, timedelta
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

print("Xcapit FHE-ML Platform - Insurance Demo")
print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)

## 2. Configuracion del Consorcio de Aseguradoras

In [ ]:
# Configuracion del dataset de seguros
INSURANCE_CONFIG = {
    "n_samples": 15000,
    "fraud_rate": 0.05,  # 5% de fraude
    "companies": [
        {"name": "Seguro Alpha", "country": "Argentina", "samples": 6000},
        {"name": "Seguro Beta", "country": "Chile", "samples": 5000},
        {"name": "Seguro Gamma", "country": "Mexico", "samples": 4000},
    ],
    "features": [
        {"name": "claim_amount", "type": "float", "min": 100, "max": 100000},
        {"name": "policy_age_months", "type": "int", "min": 1, "max": 240},
        {"name": "customer_age", "type": "int", "min": 18, "max": 80},
        {"name": "num_previous_claims", "type": "int", "min": 0, "max": 20},
        {"name": "days_to_report", "type": "int", "min": 0, "max": 365},
        {"name": "claim_type", "type": "category", "values": ["collision", "theft", "injury", "property", "other"]},
        {"name": "policy_type", "type": "category", "values": ["basic", "standard", "premium"]},
        {"name": "region", "type": "category", "values": ["north", "south", "east", "west", "central"]},
        {"name": "witnesses_present", "type": "bool", "true_ratio": 0.3},
        {"name": "police_report", "type": "bool", "true_ratio": 0.6},
    ]
}

print("Configuracion Insurance:")
print(f"  Total muestras: {INSURANCE_CONFIG['n_samples']:,}")
print(f"  Tasa de fraude: {INSURANCE_CONFIG['fraud_rate']*100}%")
print(f"  Aseguradoras participantes: {len(INSURANCE_CONFIG['companies'])}")
print(f"  Features: {len(INSURANCE_CONFIG['features'])}")
print("\nAseguradoras:")
for company in INSURANCE_CONFIG['companies']:
    print(f"   {company['name']} ({company['country']}): {company['samples']:,} reclamos")

## 3. Generacion de Datos de Reclamos de Seguros

In [ ]:
def generate_insurance_data(config: dict, seed: int = 42) -> pd.DataFrame:
    """Genera datos sinteticos de reclamos de seguros."""
    np.random.seed(seed)
    
    n = config["n_samples"]
    
    # Generar features basicas con sklearn para correlaciones
    X, y = make_classification(
        n_samples=n,
        n_features=10,
        n_informative=7,
        n_redundant=2,
        n_classes=2,
        weights=[1 - config["fraud_rate"], config["fraud_rate"]],
        random_state=seed
    )
    
    df = pd.DataFrame()
    
    # Claim amount: $100 - $100,000
    df['claim_amount'] = np.abs(X[:, 0]) * 15000 + 500
    df.loc[y == 1, 'claim_amount'] *= np.random.uniform(1.5, 4, y.sum())  # Fraudes tienden a ser mayores
    df['claim_amount'] = df['claim_amount'].clip(100, 100000).round(2)
    
    # Policy age: 1-240 months
    df['policy_age_months'] = np.abs(X[:, 1]) * 60 + 12
    df.loc[y == 1, 'policy_age_months'] = np.random.randint(1, 24, y.sum())  # Fraudes en polizas nuevas
    df['policy_age_months'] = df['policy_age_months'].clip(1, 240).astype(int)
    
    # Customer age: 18-80
    df['customer_age'] = np.abs(X[:, 2]) * 20 + 35
    df['customer_age'] = df['customer_age'].clip(18, 80).astype(int)
    
    # Previous claims: 0-20
    df['num_previous_claims'] = np.random.poisson(1.5, n)
    df.loc[y == 1, 'num_previous_claims'] = np.random.poisson(4, y.sum())  # Fraudes tienen mas historial
    df['num_previous_claims'] = df['num_previous_claims'].clip(0, 20)
    
    # Days to report: 0-365
    df['days_to_report'] = np.random.exponential(10, n)
    df.loc[y == 1, 'days_to_report'] = np.random.exponential(45, y.sum())  # Fraudes reportan tarde
    df['days_to_report'] = df['days_to_report'].clip(0, 365).astype(int)
    
    # Claim type
    claim_types = config['features'][5]['values']
    df['claim_type'] = np.random.choice(claim_types, n, p=[0.30, 0.15, 0.25, 0.20, 0.10])
    df.loc[y == 1, 'claim_type'] = np.random.choice(['theft', 'injury'], y.sum())  # Fraudes en theft/injury
    
    # Policy type
    policy_types = config['features'][6]['values']
    df['policy_type'] = np.random.choice(policy_types, n, p=[0.40, 0.45, 0.15])
    
    # Region
    regions = config['features'][7]['values']
    df['region'] = np.random.choice(regions, n)
    
    # Witnesses present (fraudes suelen no tener testigos)
    df['witnesses_present'] = np.random.random(n) < 0.35
    df.loc[y == 1, 'witnesses_present'] = np.random.random(y.sum()) < 0.1
    
    # Police report (fraudes menos probable que tengan reporte)
    df['police_report'] = np.random.random(n) < 0.65
    df.loc[y == 1, 'police_report'] = np.random.random(y.sum()) < 0.3
    
    # Target
    df['is_fraudulent'] = y
    
    # Assign to companies
    company_labels = []
    for company in config['companies']:
        company_labels.extend([company['name']] * company['samples'])
    df['company'] = company_labels[:n]
    
    # Generate claim IDs
    df['claim_id'] = [f"CLM-{hashlib.sha256(str(i).encode()).hexdigest()[:8].upper()}" for i in range(n)]
    
    return df

# Generar datos
df_claims = generate_insurance_data(INSURANCE_CONFIG)

print("\nDataset Generado:")
print(f"  Shape: {df_claims.shape}")
print(f"  Fraudes: {df_claims['is_fraudulent'].sum()} ({df_claims['is_fraudulent'].mean()*100:.2f}%)")
print(f"\nDistribucion por aseguradora:")
print(df_claims.groupby('company')['is_fraudulent'].agg(['count', 'sum', 'mean']).round(4))

In [ ]:
# Vista previa de los datos
print("Vista previa de reclamos:")
print("="*100)
display_cols = ['claim_id', 'claim_amount', 'claim_type', 'policy_age_months', 'days_to_report', 
                'witnesses_present', 'police_report', 'is_fraudulent', 'company']
df_claims[display_cols].head(10)

## 4. Analisis Exploratorio de Datos

In [ ]:
print("ANALISIS DE PATRONES DE FRAUDE")
print("=" * 60)

# Estadisticas por tipo de reclamo
print("\nTasa de fraude por tipo de reclamo:")
print("-" * 40)
fraud_by_type = df_claims.groupby('claim_type')['is_fraudulent'].agg(['count', 'sum', 'mean'])
fraud_by_type['fraud_rate_%'] = (fraud_by_type['mean'] * 100).round(2)
print(fraud_by_type)

# Estadisticas por monto
print("\nMonto promedio por tipo:")
print("-" * 40)
print(f"  Reclamos legitimos: ${df_claims[df_claims['is_fraudulent']==0]['claim_amount'].mean():,.2f}")
print(f"  Reclamos fraudulentos: ${df_claims[df_claims['is_fraudulent']==1]['claim_amount'].mean():,.2f}")

# Dias para reportar
print("\nDias promedio para reportar:")
print("-" * 40)
print(f"  Reclamos legitimos: {df_claims[df_claims['is_fraudulent']==0]['days_to_report'].mean():.1f} dias")
print(f"  Reclamos fraudulentos: {df_claims[df_claims['is_fraudulent']==1]['days_to_report'].mean():.1f} dias")

## 5. Encriptacion con FHE (CKKS)

In [ ]:
def simulate_encryption(data: np.ndarray) -> dict:
    """Simula la encriptacion FHE para demo."""
    data_bytes = data.tobytes()
    cipher_hash = hashlib.sha256(data_bytes).hexdigest()
    
    return {
        "ciphertext_preview": f"0x{cipher_hash[:64]}",
        "original_shape": data.shape,
        "encrypted_size_kb": len(data_bytes) * 100 // 1024,
        "scheme": "CKKS",
        "security_bits": 128,
        "poly_modulus_degree": 8192
    }

# Preparar datos para encriptacion
feature_cols = ['claim_amount', 'policy_age_months', 'customer_age', 
                'num_previous_claims', 'days_to_report']

# Encoding de categoricas
df_encoded = df_claims.copy()
le_claim = LabelEncoder()
le_policy = LabelEncoder()
le_region = LabelEncoder()

df_encoded['claim_type_enc'] = le_claim.fit_transform(df_encoded['claim_type'])
df_encoded['policy_type_enc'] = le_policy.fit_transform(df_encoded['policy_type'])
df_encoded['region_enc'] = le_region.fit_transform(df_encoded['region'])
df_encoded['witnesses_present'] = df_encoded['witnesses_present'].astype(int)
df_encoded['police_report'] = df_encoded['police_report'].astype(int)

feature_cols_full = feature_cols + ['claim_type_enc', 'policy_type_enc', 'region_enc', 
                                     'witnesses_present', 'police_report']

X = df_encoded[feature_cols_full].values
y = df_encoded['is_fraudulent'].values

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Datos preparados para encriptacion:")
print(f"  X shape: {X_scaled.shape}")
print(f"  y shape: {y.shape}")
print(f"  Features: {feature_cols_full}")

In [ ]:
# Demostrar diferencia entre plaintext y ciphertext
print("COMPARACION: PLAINTEXT vs CIPHERTEXT")
print("=" * 60)

# Muestra de datos en plaintext
sample_idx = 0
sample_plaintext = df_claims.iloc[sample_idx]

print("\n[PLAINTEXT] - Datos expuestos:")
print("-" * 40)
print(f"  Claim ID: {sample_plaintext['claim_id']}")
print(f"  Monto: ${sample_plaintext['claim_amount']:,.2f}")
print(f"  Tipo: {sample_plaintext['claim_type']}")
print(f"  Edad poliza: {sample_plaintext['policy_age_months']} meses")
print(f"  Dias para reportar: {sample_plaintext['days_to_report']}")
print(f"  Testigos: {'Si' if sample_plaintext['witnesses_present'] else 'No'}")
print(f"  Reporte policial: {'Si' if sample_plaintext['police_report'] else 'No'}")
print(f"  Es fraude: {'SI' if sample_plaintext['is_fraudulent'] else 'No'}")

# Simular encriptacion
encrypted_info = simulate_encryption(X_scaled[sample_idx:sample_idx+1])

print("\n[CIPHERTEXT] - Datos protegidos:")
print("-" * 40)
print(f"  Datos: {encrypted_info['ciphertext_preview'][:32]}...")
print(f"  Tamano: ~{encrypted_info['encrypted_size_kb']} KB")
print(f"  Esquema: {encrypted_info['scheme']}")
print(f"  Seguridad: {encrypted_info['security_bits']} bits")
print(f"  Poly degree: {encrypted_info['poly_modulus_degree']}")

print("\n" + "=" * 60)
print("Los datos del cliente estan completamente protegidos!")

## 6. Contribuciones de las Aseguradoras (Consorcio)

In [ ]:
print("CONTRIBUCIONES DEL CONSORCIO")
print("=" * 60)

contributions = []
for company in INSURANCE_CONFIG['companies']:
    company_mask = df_claims['company'] == company['name']
    company_data = df_encoded[company_mask][feature_cols_full].values
    company_labels = df_claims[company_mask]['is_fraudulent'].values
    
    # Simular hash de contribucion
    data_hash = hashlib.sha256(company_data.tobytes()).hexdigest()[:32]
    
    # Estadisticas adicionales
    avg_claim = df_claims[company_mask]['claim_amount'].mean()
    
    contribution = {
        "company": company['name'],
        "country": company['country'],
        "records": len(company_data),
        "fraud_count": company_labels.sum(),
        "fraud_rate": company_labels.mean() * 100,
        "avg_claim": avg_claim,
        "data_hash": data_hash
    }
    contributions.append(contribution)
    
    print(f"\n{company['name']} ({company['country']}):")
    print(f"   Reclamos: {contribution['records']:,}")
    print(f"   Fraudes: {contribution['fraud_count']} ({contribution['fraud_rate']:.2f}%)")
    print(f"   Monto promedio: ${contribution['avg_claim']:,.2f}")
    print(f"   Hash encriptado: {contribution['data_hash']}...")

print("\n" + "=" * 60)
total_records = sum(c['records'] for c in contributions)
total_fraud = sum(c['fraud_count'] for c in contributions)
total_amount = df_claims['claim_amount'].sum()
print(f"Total consorcio: {total_records:,} reclamos, {total_fraud} fraudes")
print(f"Monto total: ${total_amount:,.2f}")

## 7. Votacion Commit-Reveal (Gobernanza)

In [ ]:
print("VOTACION COMMIT-REVEAL")
print("=" * 60)
print("Propuesta: Entrenar modelo de deteccion de fraude en reclamos")
print("Quorum requerido: 51%")

proposal_id = hashlib.sha256(b"TRAIN_INSURANCE_FRAUD_MODEL_2025").hexdigest()

# Fase 1: Commit
print("\n[FASE 1: COMMIT] - Votos ocultos")
print("-" * 40)

votes_secret = {}
commitments = {}

for company in INSURANCE_CONFIG['companies']:
    vote = True  # Todos votan SI
    salt = secrets.token_bytes(32)
    
    commitment = hashlib.sha256(
        proposal_id.encode() + bytes([vote]) + salt
    ).hexdigest()
    
    votes_secret[company['name']] = (vote, salt)
    commitments[company['name']] = commitment
    
    print(f"  {company['name']}: 0x{commitment[:24]}... (voto oculto)")

# Fase 2: Reveal
print("\n[FASE 2: REVEAL] - Votos verificados")
print("-" * 40)

yes_votes = 0
for company_name, (vote, salt) in votes_secret.items():
    expected = hashlib.sha256(
        proposal_id.encode() + bytes([vote]) + salt
    ).hexdigest()
    
    verified = expected == commitments[company_name]
    
    if verified and vote:
        yes_votes += 1
    
    status = "VERIFICADO" if verified else "INVALIDO"
    vote_str = "SI" if vote else "NO"
    print(f"  {company_name}: {vote_str} - {status}")

print("\n" + "=" * 60)
approval_pct = (yes_votes / len(INSURANCE_CONFIG['companies'])) * 100
print(f"Resultado: {yes_votes}/{len(INSURANCE_CONFIG['companies'])} ({approval_pct:.0f}%)")
print(f"Estado: {'APROBADO - Entrenamiento autorizado' if approval_pct >= 51 else 'RECHAZADO'}")

## 8. Entrenamiento del Modelo

In [ ]:
print("ENTRENAMIENTO DEL MODELO")
print("=" * 60)

# Split de datos
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Datos de entrenamiento: {len(X_train):,} muestras")
print(f"Datos de prueba: {len(X_test):,} muestras")
print(f"Fraudes en train: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Fraudes en test: {y_test.sum()} ({y_test.mean()*100:.2f}%)")

# Entrenar modelo (Random Forest para mejor deteccion de fraude)
print("\nEntrenando Random Forest Classifier...")
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Metricas
accuracy = accuracy_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_prob)

print("\nEntrenamiento completado!")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"AUC-ROC: {auc_roc:.4f}")

In [ ]:
# Reporte detallado
print("REPORTE DE CLASIFICACION")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Legitimo', 'Fraude']))

print("\nMATRIZ DE CONFUSION")
print("-" * 40)
cm = confusion_matrix(y_test, y_pred)
print(f"                  Predicho")
print(f"               Legitimo  Fraude")
print(f"Real Legitimo    {cm[0,0]:5d}    {cm[0,1]:5d}")
print(f"Real Fraude      {cm[1,0]:5d}    {cm[1,1]:5d}")

In [ ]:
# Feature importance
print("IMPORTANCIA DE FEATURES")
print("=" * 60)

feature_importance = pd.DataFrame({
    'feature': feature_cols_full,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 5 features mas importantes:")
for i, row in feature_importance.head(5).iterrows():
    bar = '#' * int(row['importance'] * 50)
    print(f"  {row['feature']:<25} {row['importance']:.4f} {bar}")

## 9. Predicciones en Tiempo Real

In [ ]:
# Crear nuevos reclamos para evaluacion
new_claims = pd.DataFrame({
    'claim_id': ['NEW-001', 'NEW-002', 'NEW-003', 'NEW-004', 'NEW-005', 'NEW-006'],
    'claim_amount': [2500.00, 45000.00, 850.00, 78000.00, 1200.00, 55000.00],
    'policy_age_months': [36, 3, 120, 6, 48, 2],
    'customer_age': [45, 28, 62, 35, 55, 30],
    'num_previous_claims': [1, 5, 0, 8, 2, 6],
    'days_to_report': [2, 45, 1, 90, 5, 60],
    'claim_type_enc': [0, 1, 2, 1, 3, 1],  # collision, theft, injury, theft, property, theft
    'policy_type_enc': [1, 0, 2, 0, 1, 0],  # standard, basic, premium, basic, standard, basic
    'region_enc': [2, 0, 1, 3, 2, 4],
    'witnesses_present': [1, 0, 1, 0, 1, 0],
    'police_report': [1, 0, 1, 0, 1, 0],
    'description': [
        'Accidente de trafico con testigos',
        'Robo de vehiculo sin evidencia',
        'Lesion menor en trabajo',
        'Robo de alto valor, sin reporte policial',
        'Dano a propiedad con documentacion',
        'Robo reciente, poliza nueva'
    ]
})

print("EVALUACION DE NUEVOS RECLAMOS")
print("=" * 80)

# Preparar y predecir
X_new = new_claims[feature_cols_full].values
X_new_scaled = scaler.transform(X_new)

predictions = model.predict(X_new_scaled)
probabilities = model.predict_proba(X_new_scaled)[:, 1]

print(f"{'ID':<10} {'Monto':>12} {'Poliza':>8} {'Dias':>6} {'Riesgo':>8} {'Resultado':>12}")
print("-" * 80)

for i, row in new_claims.iterrows():
    claim_id = row['claim_id']
    amount = row['claim_amount']
    policy_months = row['policy_age_months']
    days = row['days_to_report']
    risk = probabilities[i] * 100
    result = "FRAUDE" if predictions[i] == 1 else "Legitimo"
    flag = " " if predictions[i] == 0 else "!!!"
    
    print(f"{claim_id:<10} ${amount:>10,.2f} {policy_months:>6}m {days:>5}d {risk:>7.1f}% {result:>10} {flag}")

print("\n" + "-" * 80)
flagged = predictions.sum()
flagged_amount = new_claims.loc[predictions == 1, 'claim_amount'].sum()
print(f"Reclamos marcados como fraude: {flagged}")
print(f"Monto en riesgo: ${flagged_amount:,.2f}")

In [ ]:
# Detalles de reclamos sospechosos
print("\nDETALLE DE RECLAMOS SOSPECHOSOS")
print("=" * 60)

for i, row in new_claims.iterrows():
    if predictions[i] == 1:
        print(f"\n{row['claim_id']}: {row['description']}")
        print("-" * 40)
        print(f"   Monto: ${row['claim_amount']:,.2f}")
        print(f"   Edad poliza: {row['policy_age_months']} meses")
        print(f"   Dias para reportar: {row['days_to_report']}")
        print(f"   Reclamos previos: {row['num_previous_claims']}")
        print(f"   Testigos: {'Si' if row['witnesses_present'] else 'No'}")
        print(f"   Reporte policial: {'Si' if row['police_report'] else 'No'}")
        print(f"   Probabilidad de fraude: {probabilities[i]*100:.1f}%")
        print(f"   Recomendacion: Investigacion detallada requerida")

## 10. Resumen y Garantias de Privacidad

In [ ]:
print("RESUMEN DEL DEMO INSURANCE")
print("=" * 60)

print("""
GARANTIAS DE PRIVACIDAD
-----------------------
 Los datos de cada aseguradora NUNCA se comparten en plaintext
 Toda la informacion esta encriptada con CKKS (128-bit)
 El modelo se entrena sobre datos encriptados
 Solo el cliente puede desencriptar sus resultados
 Votacion commit-reveal evita manipulacion
 Audit trail completo en Arbitrum blockchain

METRICAS DEL CONSORCIO
----------------------
""")
print(f"  Aseguradoras participantes: {len(INSURANCE_CONFIG['companies'])}")
print(f"  Total reclamos: {INSURANCE_CONFIG['n_samples']:,}")
print(f"  Fraudes detectados: {y.sum()}")
print(f"  Accuracy del modelo: {accuracy*100:.2f}%")
print(f"  AUC-ROC: {auc_roc:.4f}")
print(f"  Monto total en reclamos: ${df_claims['claim_amount'].sum():,.2f}")

print("""
CUMPLIMIENTO REGULATORIO
------------------------
 Ley de Proteccion de Datos Personales (Argentina)
 Ley 19.628 (Chile)
 LFPDPPP (Mexico)
 GDPR (Operaciones internacionales)

SMART CONTRACTS DESPLEGADOS
---------------------------
Red: Arbitrum Sepolia (Testnet)
""")
print("  Governance:    0xda52326d106A91A1F22A0c41Be2dc1F531C01F11")
print("  Registry:      0x1296cCeF7803Bff51FB690afCFc586E7012417b8")
print("  Verifier:      0xa5f04E0aefe55173C91b949Aa2385f0228dd2921")
print("\nExplorador: https://sepolia.arbiscan.io")

---

## Proximos Pasos

1. **Probar con datos reales**: Conectar con la API en produccion
2. **Unirse a un consorcio existente**: Dashboard > Consorcios > Buscar
3. **Explorar otros verticales**: Fintech, Healthcare, Retail, Government

**Documentacion**: https://apifhe.xcapit.com/api/v2/docs/